# Lab 7: Planning and Learning Integration

**MSDS 684 — Reinforcement Learning · Regis University**

**Author:** Morgan Cooper  
**Reading:** Sutton & Barto (2018), *Reinforcement Learning: An Introduction*, Chapter 8

---

## Objective

Implement Dyna-Q on Gymnasium's **Taxi-v3** to study how integrating a learned model with direct RL changes sample efficiency, then extend the agent to handle a non-stationary environment (Dyna-Q+) and a focused planning order (prioritized sweeping).

## What this notebook delivers

1. **Direct RL** baseline — pure Q-learning on Taxi-v3 (NumPy Q-table).
2. **Model Learning** — deterministic tabular model stored as `model[(s,a)] = (r, s')` in a Python dict.
3. **Planning** — *n* simulated Q-updates per real env step using sampled `(s,a)` from the model.
4. Sample-efficiency comparison: pure Q-learning (n=0) vs. Dyna-Q with **n ∈ {5, 10, 50}**.
5. **Dyna-Q+** with the `κ√τ` exploration bonus on a wrapper that changes the env after 1000 steps.
6. **Prioritized sweeping** with a `heapq`-backed priority queue keyed on TD-error magnitude.
7. Final synthesis: when models help, when they hurt, and what model error costs.

## Mapping to the four report sections

| Notebook part | Feeds into report section |
|---|---|
| Part 1–2 (concept + env) | Section 1: Project Overview |
| Part 3–9 (implementation, experiments, figures) | Section 2: Deliverables |
| Part 10 (debugging notes inline) | Section 3: AI Use Reflection |
| Part 11 (synthesis bullets) | Section 4: Speaker Notes |


---

## Part 1 — Setup and Imports

**Goal:** establish a reproducible environment for every experiment in the notebook.

**Steps:**

- Import `numpy`, `matplotlib`, `gymnasium`, `heapq`, `random`, `collections.defaultdict`, `dataclasses`.
- Set a master RNG seed and note that each experiment will use a list of seeds (≥ 10, ideally 30) for averaging.
- Configure matplotlib defaults (figure size, font size, grid).
- Create a `figures/` directory if it does not already exist — every plot saved here will be embedded in the PDF report.
- Capture library versions in a short markdown line so the notebook is self-documenting (Gymnasium has API drift between versions; record what was used).


---

## Part 2 — Environment: Taxi-v3

**Goal:** describe the MDP precisely so Section 1 of the report has the required environment block.

**Steps:**

- Instantiate `gymnasium.make('Taxi-v3')` and confirm the API (`reset() -> (obs, info)`, `step() -> (obs, reward, terminated, truncated, info)`).
- Record the spec:
  - **State space:** 500 discrete states (taxi row × col × passenger location × destination).
  - **Action space:** 6 discrete actions — south, north, east, west, pickup, dropoff.
  - **Rewards:** −1 per step, +20 successful dropoff, −10 illegal pickup/dropoff.
  - **Termination:** successful dropoff; episodes also truncate after a step cap.
- Run a single random-policy episode just to verify the API contract; do **not** include this output in the final PDF.
- Note Taxi's properties relevant to Dyna-Q: small, discrete, deterministic transitions and rewards → a tabular dict model is exact and cheap.


---

## Part 3 — Direct RL Baseline (Q-learning, n = 0)

**Goal:** establish a model-free baseline that Dyna-Q must beat on sample efficiency.

**Steps:**

- Initialize `Q = np.zeros((n_states, n_actions))`.
- Implement `epsilon_greedy(Q, s, epsilon)` — break ties randomly.
- Train loop:
  - For each episode, step until `terminated or truncated`.
  - Apply Q-learning update: `Q[s,a] += alpha * (r + gamma * max(Q[s']) - Q[s,a])`.
  - Track per-episode return and per-real-step cumulative reward (Dyna-Q comparisons are plotted vs. real env steps, not episodes).
- Hyperparameters to record: `alpha`, `gamma`, `epsilon` (and any decay), episode cap, total real steps, number of seeds.
- Run across multiple seeds and store mean ± 95% CI of returns and cumulative reward.

---

## Part 4 — Dyna-Q: Three Processes in One Loop

**Goal:** implement the three integrated components from the lab brief.

**Steps:**

1. **Direct RL** — same Q-learning update as Part 3, applied to *real* `(s, a, r, s')` transitions.
2. **Model Learning** — after every real `env.step()`, write `model[(s, a)] = (r, s')`. Deterministic Taxi means each key has a single ground-truth target.
3. **Planning** — for `n` planning iterations per real step:
   - Sample `(s, a)` uniformly from `model.keys()`.
   - Look up `(r, s') = model[(s, a)]`.
   - Apply the same Q-learning update on this *simulated* experience.

**Implementation notes:**

- Keep `model` as a plain `dict` so `random.choice(list(model.keys()))` is straightforward; cache the key list and refresh only when new keys are added if profiling shows it matters.
- Wrap the agent in a class (e.g. `DynaQAgent`) so Dyna-Q+ and prioritized sweeping in later parts can subclass / share code.
- Verify with `n = 0` reproduces the Part 3 baseline exactly given the same seed.

---

## Part 5 — Sample-Efficiency Experiment (n ∈ {0, 5, 10, 50})

**Goal:** produce the headline comparison the lab brief calls for.

**Steps:**

- Fix `alpha`, `gamma`, `epsilon` across all four configurations so the only changing variable is `n`.
- For each `n` in `{0, 5, 10, 50}` and each seed, run training and log:
  - cumulative reward as a function of **real** env steps,
  - per-episode return,
  - real env steps to first reach an "optimal" performance threshold (define the threshold explicitly — e.g. mean return over last 100 episodes ≥ X).
- Aggregate across seeds → mean and 95% CI.
- Save raw arrays (e.g. `.npz`) so figures can be rebuilt without retraining.

---

## Part 6 — Visualizations for the Static Environment

**Goal:** generate the three figures the brief explicitly asks for.

**Plots to save to `figures/`:**

1. **Cumulative reward over real time steps** — one curve per `n`, with shaded 95% CI band.
2. **Episodes until optimal performance** — bar or box plot across `n` values, showing distribution over seeds.
3. **Sample efficiency** — real env interactions needed to reach the threshold, one bar per `n`.

**Caption discipline:**

- Captions must be **interpretive**, not descriptive. State *what* the figure shows AND *why* it matters (cite S&B Ch. 8 where relevant).
- Each figure goes into the PDF report; do not include console output dumps.

---

## Part 7 — Non-Stationary Environment Wrapper

**Goal:** create a controlled environmental change at step 1000 to motivate Dyna-Q+.

**Steps:**

- Subclass `gymnasium.Wrapper` (e.g. `ChangingTaxi`) that tracks total real steps internally.
- After 1000 steps, apply a structural change. Concrete options (pick one, document it):
  - block one of the previously legal moves between two cells (analogous to a wall change),
  - move the destination square so the optimal route shifts,
  - increase the magnitude of the dropoff reward at a specific square so a previously suboptimal route becomes optimal (the harder "opened shortcut" case from S&B §8.3).
- Document the exact mechanism in the markdown immediately above the wrapper — the report's analysis depends on knowing what "changed" means.
- Verify by stepping the wrapped env around step 1000 and confirming pre/post transitions differ.

---

## Part 8 — Dyna-Q+ with the κ√τ Exploration Bonus

**Goal:** detect and adapt to environment change faster than vanilla Dyna-Q.

**Steps:**

- Add a second dictionary `time_since[(s, a)]` initialized to 0.
- After every real step, increment `time_since[(s, a)]` for all known `(s, a)` and reset the just-visited pair to 0.
- During *planning* (not direct RL), augment the simulated reward with `kappa * sqrt(time_since[(s, a)])`.
- Hyperparameter sweep on `kappa` is optional but useful: too small → no recovery, too large → ignores learned values.
- Note (S&B §8.3): the bonus only enters planning updates, not real-experience updates.

---

## Part 9 — Dyna-Q vs. Dyna-Q+ on the Changing Environment

**Goal:** show, with numbers, that Dyna-Q+ recovers faster after the step-1000 change.

**Steps:**

- Run both agents on `ChangingTaxi` with the same `n`, `alpha`, `gamma`, `epsilon`, and seed list.
- Plot cumulative reward vs. real steps with a vertical line at the change-point.
- Quantify the **adaptation gap**: real steps after change until each agent re-reaches its pre-change performance.
- Save the figure to `figures/` and write an interpretive caption that ties recovery speed to the κ√τ mechanism.

---

## Part 10 — Prioritized Sweeping

**Goal:** replace uniform random planning with TD-error-prioritized planning.

**Steps:**

- Maintain a min-heap via `heapq` storing `(-priority, counter, s, a)` tuples (negate priority to get max-heap behavior; the integer counter breaks ties so `heapq` never compares states).
- After each real step, compute the TD-error magnitude `p = |r + gamma * max_a' Q(s', a') − Q(s, a)|`. If `p > theta`, push the pair onto the queue.
- Maintain a **predecessors** dict: `pred[s']` = set of `(s, a)` pairs that have ever transitioned into `s'`.
- Planning loop: pop the highest-priority pair, perform the Q-update, then for every predecessor compute its priority and push if it exceeds `theta`.
- Compare against uniform-random planning at matched `n` and report learning-speed differences (real env steps to reach threshold).
- Save a comparison figure with an interpretive caption (S&B §8.4: focusing updates near recently changed values propagates information faster).

---

## Part 11 — Synthesis

**Goal:** answer the four synthesis questions from the lab brief; this content seeds the report's analysis prose and Speaker Notes.

**Questions to answer in markdown:**

1. **When to use model-based vs. model-free?** — cost of real experience, environment determinism, stationarity, state-space size.
2. **How does sample complexity compare?** — reference the Part 5/6 numbers.
3. **Computational trade-offs?** — wall-clock cost per real step grows with `n`; planning is cheap on tabular Taxi but expensive on large/continuous problems.
4. **How do model errors affect performance?** — reference Part 9 (stale model after change) and the prioritized-sweeping behavior.

**Connection cells (for Speaker Notes):**

- Bridge to deep model-based RL (world models, MuZero) and modern libraries (Stable-Baselines3, RLlib, CleanRL).
- Identify one design decision worth highlighting in the 5-minute talk.

---

## Part 12 — Optional Extension: Learned Neural Network Dynamics Model

**Goal (only if time allows):** replace the dictionary model with a PyTorch network that predicts `(s', r)` from `(s, a)` on a continuous environment such as `MountainCar-v0`.

**Steps (sketch):**

- Collect a buffer of real transitions.
- Train a small MLP to predict next state and reward; track validation MSE.
- Plug the learned model into the planning loop and observe how prediction error compounds during multi-step rollouts.
- Briefly note how this would interface with Stable-Baselines3 / RLlib / CleanRL.

---

## Part 13 — Debugging Log (for the AI Use Reflection)

**Goal:** keep a running record while implementing — this is where Section 3 of the report comes from.

**For each non-trivial bug, capture:**

- The error or unexpected behavior (paste the traceback / one-line symptom).
- The follow-up prompt I sent to the AI.
- A summary of the AI's response.
- Whether it worked the first time, and how I verified the fix.

**Plausible Lab 7 bug surface (watch for these):**

- Stale `(r, s')` in the model after the wrapper changes the env (motivates Dyna-Q+).
- Off-by-one in `time_since[(s, a)]` increments.
- `heapq` comparing states because two entries share a priority — fix with a monotonic counter.
- Predecessors set growing without bound — confirm it is keyed correctly.
- Q-learning update on simulated experience using a stale `max(Q[s'])` (e.g. computed before the update).
- Gymnasium API drift (`reset()` returning `(obs, info)`; `step()` returning the 5-tuple).

**Reminder:** the AI Use Reflection is 35% of the grade. Aim for **3+ concrete debugging cycles** with specific errors and resolutions.

---

## Pre-Submission Checklist

From the Lab Directions PDF:

- [ ] PDF filename is `Cooper_Morgan_Lab7.pdf`.
- [ ] GitHub repository URL appears at the top of Section 2 and is clickable.
- [ ] Project Overview is 400–500 words.
- [ ] No raw code listings or console output anywhere in the PDF.
- [ ] 2–4 visualizations embedded with **interpretive** captions.
- [ ] AI Use Reflection is 250–350 words with specific iteration examples.
- [ ] Speaker Notes are 5–7 bullets covering ~5 minutes.
- [ ] All four sections present and clearly labeled.
- [ ] GitHub repo: working code, `requirements.txt` or `environment.yml`, README, organized files (not one giant script), commit history that matches the AI reflection narrative.